# Notebook 03 — Limpeza e Clustering de Temas

**Sprint 2 — Lei e Política**

## Conteúdo

1. Carregar o corpus de ementas (Câmara + Senado) do Supabase
2. Limpeza do texto (stopwords PT + jurídicas, remoção de acentos)
3. TF-IDF + K-Means (k=6..10), escolha de k por silhouette
4. Top-10 termos por cluster
5. **Pausa para nomeação humana** dos clusters em linguagem cidadã
6. Update de `tema_cluster` e `tema_cidadao` em `proposicoes`

> O texto legislativo circula pelas duas casas; agrupar as ementas por tema dá o
> rótulo usado na predição de voto (Sprint 3) e na interface cidadã (Sprint 4).

In [2]:
import sys
sys.path.insert(0, '..')

import re
import unicodedata
import logging

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

from src.db import buscar_todos, upsert_proposicoes

logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')
log = logging.getLogger('03_clustering')
print('Módulos carregados.')

Módulos carregados.


## 1. Carregar corpus de ementas

Lê todas as proposições (Câmara + Senado) do Supabase e descarta ementas vazias ou curtas demais para gerar sinal.

In [3]:
props = buscar_todos('proposicoes', 'id,id_externo,casa,ementa')
df = pd.DataFrame(props)
print(f'Proposições no banco: {len(df)}')

df = df[df['ementa'].notna()].copy()
df['ementa'] = df['ementa'].astype(str)
df = df[df['ementa'].str.len() >= 10].copy()

print(f'Com ementa válida (>= 10 chars): {len(df)}')
print(df['casa'].value_counts())

2026-06-23 22:14:06,907 [INFO] HTTP Request: GET https://woxluznsfqskopzzjqtf.supabase.co/rest/v1/proposicoes?select=id%2Cid_externo%2Ccasa%2Cementa&offset=0&limit=1000 "HTTP/2 200 OK"
2026-06-23 22:14:08,221 [INFO] HTTP Request: GET https://woxluznsfqskopzzjqtf.supabase.co/rest/v1/proposicoes?select=id%2Cid_externo%2Ccasa%2Cementa&offset=1000&limit=1000 "HTTP/2 200 OK"
2026-06-23 22:14:09,030 [INFO] HTTP Request: GET https://woxluznsfqskopzzjqtf.supabase.co/rest/v1/proposicoes?select=id%2Cid_externo%2Ccasa%2Cementa&offset=2000&limit=1000 "HTTP/2 200 OK"
2026-06-23 22:14:10,154 [INFO] HTTP Request: GET https://woxluznsfqskopzzjqtf.supabase.co/rest/v1/proposicoes?select=id%2Cid_externo%2Ccasa%2Cementa&offset=3000&limit=1000 "HTTP/2 200 OK"
2026-06-23 22:14:10,876 [INFO] HTTP Request: GET https://woxluznsfqskopzzjqtf.supabase.co/rest/v1/proposicoes?select=id%2Cid_externo%2Ccasa%2Cementa&offset=4000&limit=1000 "HTTP/2 200 OK"
2026-06-23 22:14:11,916 [INFO] HTTP Request: GET https://woxluz

Proposições no banco: 22041
Com ementa válida (>= 10 chars): 22041
casa
camara    11697
senado    10344
Name: count, dtype: int64


## 2. Limpeza do texto

Minúsculas → remoção de acentos → só letras → descarte de tokens curtos e stopwords (PT + jurídicas). A lista é embutida (offline e determinística); se o `nltk` já tiver as stopwords baixadas, elas são incorporadas, mas nunca há dependência de rede.

In [4]:
def remover_acentos(texto):
    nfkd = unicodedata.normalize('NFKD', texto)
    return ''.join(c for c in nfkd if not unicodedata.combining(c))

# Stopwords PT comuns (já sem acento — o texto também é desacentuado antes de comparar)
STOPWORDS_PT = {
    'a', 'o', 'as', 'os', 'um', 'uma', 'uns', 'umas', 'de', 'do', 'da', 'dos',
    'das', 'em', 'no', 'na', 'nos', 'nas', 'por', 'pelo', 'pela', 'pelos',
    'pelas', 'com', 'sem', 'sob', 'sobre', 'para', 'pra', 'ate', 'entre',
    'contra', 'desde', 'e', 'ou', 'mas', 'que', 'se', 'como', 'quando',
    'porque', 'pois', 'ja', 'nao', 'sim', 'ao', 'aos', 'este', 'esta', 'estes',
    'estas', 'esse', 'essa', 'esses', 'essas', 'isto', 'isso', 'aquele',
    'aquela', 'aquilo', 'seu', 'sua', 'seus', 'suas', 'dele', 'dela', 'deles',
    'delas', 'meu', 'minha', 'nosso', 'nossa', 'ele', 'ela', 'eles', 'elas',
    'eu', 'tu', 'voce', 'nos', 'vos', 'lhe', 'lhes', 'me', 'te', 'foi', 'ser',
    'sao', 'era', 'sera', 'tem', 'ter', 'havia', 'mais', 'menos', 'muito',
    'pouco', 'todo', 'toda', 'todos', 'todas', 'outro', 'outra', 'outros',
    'outras', 'mesmo', 'mesma', 'qual', 'quais', 'onde', 'seja', 'sejam',
    'tambem', 'apenas', 'cada', 'ainda', 'assim', 'entao',
}

# Termos quase onipresentes em ementas — não distinguem temas
STOPWORDS_JURIDICAS = {
    'lei', 'leis', 'art', 'arts', 'artigo', 'artigos', 'paragrafo', 'inciso',
    'alinea', 'dispoe', 'dispor', 'altera', 'alteracao', 'alterar', 'institui',
    'instituir', 'estabelece', 'estabelecer', 'providencias', 'outras', 'revoga',
    'revogacao', 'vigencia', 'dar', 'acrescenta', 'inclui', 'inclusao',
    'modifica', 'denomina', 'denominacao', 'autoriza', 'autorizacao', 'cria',
    'criacao', 'federal', 'nacional', 'numero', 'decreto', 'medida', 'provisoria',
    'projeto', 'proposta', 'emenda', 'constituicao', 'codigo', 'normas', 'norma',
    'regula', 'regulamenta', 'define', 'fixa', 'concede', 'referente', 'relativo',
    'relativa', 'seguinte', 'seguintes', 'forma', 'âmbito', 'ambito',
}

STOPWORDS = STOPWORDS_PT | STOPWORDS_JURIDICAS

# Enriquecer com nltk se disponível — sem rede; falha silenciosa
try:
    from nltk.corpus import stopwords as _nltk_sw
    STOPWORDS |= {remover_acentos(w) for w in _nltk_sw.words('portuguese')}
    print(f'nltk stopwords incorporadas. Total: {len(STOPWORDS)} termos.')
except Exception:
    print(f'nltk indisponível — usando lista embutida ({len(STOPWORDS)} termos).')


def limpar(texto):
    texto = remover_acentos(texto.lower())
    texto = re.sub(r'[^a-z\s]', ' ', texto)
    tokens = [t for t in texto.split() if len(t) >= 3 and t not in STOPWORDS]
    return ' '.join(tokens)


df['texto_limpo'] = df['ementa'].map(limpar)
df = df[df['texto_limpo'].str.len() > 0].copy()
print(f'Documentos após limpeza: {len(df)}')
print('Exemplo:', df['texto_limpo'].iloc[0][:160])

nltk indisponível — usando lista embutida (176 termos).
Documentos após limpeza: 22040
Exemplo: organizacao basica orgaos presidencia republica ministerios


## 3. TF-IDF e escolha de k

Vetoriza o corpus limpo e varre `k=6..10` no K-Means. O melhor `k` é o de maior
silhouette (calculado sobre uma amostra de 5.000 documentos para evitar custo O(n²)).
A curva é mostrada como tabela + barra ASCII (sem matplotlib).

In [5]:
vectorizer = TfidfVectorizer(
    max_df=0.5, min_df=5, ngram_range=(1, 2), max_features=5000,
)
X = vectorizer.fit_transform(df['texto_limpo'])
print(f'Matriz TF-IDF: {X.shape[0]} documentos × {X.shape[1]} termos')

# Amostra fixa para o silhouette (evita O(n²) no corpus inteiro)
sample = min(5000, X.shape[0])

resultados = []
for k in range(6, 11):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X)
    sil = silhouette_score(X, labels, sample_size=sample, random_state=42)
    resultados.append({'k': k, 'silhouette': sil})
    print(f'k={k}  silhouette={sil:.4f}')

sil_df = pd.DataFrame(resultados)
maxs = sil_df['silhouette'].max()

print('\n=== Silhouette × k ===')
for _, row in sil_df.iterrows():
    barra = '#' * int(round(40 * row['silhouette'] / maxs)) if maxs > 0 else ''
    print(f"k={int(row['k']):2d} | {barra} {row['silhouette']:.4f}")

melhor_k = int(sil_df.loc[sil_df['silhouette'].idxmax(), 'k'])
print(f'\nMelhor k por silhouette: {melhor_k}')

Matriz TF-IDF: 22040 documentos × 5000 termos
k=6  silhouette=0.0236
k=7  silhouette=0.0299
k=8  silhouette=0.0316
k=9  silhouette=0.0307
k=10  silhouette=0.0317

=== Silhouette × k ===
k= 6 | ############################## 0.0236
k= 7 | ###################################### 0.0299
k= 8 | ######################################## 0.0316
k= 9 | ####################################### 0.0307
k=10 | ######################################## 0.0317

Melhor k por silhouette: 10


## 4. Clustering final e top-10 termos por cluster

Reajusta o K-Means com o melhor `k` e lista os termos mais característicos de cada
cluster (maiores pesos no centróide) — a base para a nomeação cidadã.

In [6]:
km = KMeans(n_clusters=melhor_k, random_state=42, n_init=10)
df['tema_cluster'] = km.fit_predict(X)

termos = vectorizer.get_feature_names_out()
ordem_centroides = km.cluster_centers_.argsort()[:, ::-1]

print(f'=== Top-10 termos por cluster (k={melhor_k}) ===\n')
for c in range(melhor_k):
    top = [termos[idx] for idx in ordem_centroides[c, :10]]
    n = int((df['tema_cluster'] == c).sum())
    print(f'Cluster {c}  ({n} proposições):')
    print('  ' + ', '.join(top) + '\n')

=== Top-10 termos por cluster (k=10) ===

Cluster 0  (422 proposições):
  destaque, votacao, votacao separado, separado, destaque votacao, requer destaque, requer, lideranca, requer lideranca, lideranca destaque

Cluster 1  (13685 proposições):
  requer, dia, publica, programa, politica, realizacao, estado, protecao, saude, brasil

Cluster 2  (1138 proposições):
  licenca, requer licenca, missao, requer, desempenhar, licenca desempenhar, desempenhar missao, missao politica, politica, parlamentar

Cluster 3  (1732 proposições):
  dezembro, transito, educacao, autista, transtorno, brasileiro, espectro, espectro autista, imposto, transtorno espectro

Cluster 4  (381 proposições):
  saude, unico saude, sistema unico, unico, sistema, saude sus, sus, atencao, tratamento, diretrizes

Cluster 5  (916 proposições):
  radiodifusao sonora, sonora, radiodifusao, servico radiodifusao, servico, frequencia, modulada, frequencia modulada, sonora frequencia, ltda

Cluster 6  (1231 proposições):
  penal

## 5. Pausa para nomeação humana

**Revise os top-termos acima** e dê a cada cluster um nome em linguagem cidadã
(ex.: `Saúde`, `Educação`, `Segurança Pública`, `Tributação`, `Meio Ambiente`).
Edite o dicionário `NOMES_CIDADAOS` na próxima célula antes de gravar.

In [7]:
NOMES_CIDADAOS = {
    0: 'Destaques e Requerimentos de Votação',   # destaque, votacao separado, lideranca
    1: 'Políticas Públicas e Programas Sociais',  # maior cluster — programas, proteção, saude, brasil
    2: 'Licenças Parlamentares',                  # licenca, missao politica, parlamentar
    3: 'Educação, Trânsito e Direitos Sociais',   # educacao, transito, autismo (espectro autista)
    4: 'Sistema Único de Saúde (SUS)',            # sus, atencao, tratamento, diretrizes
    5: 'Radiodifusão Sonora (Rádio FM/AM)',       # radiodifusao sonora, frequencia modulada, ltda
    6: 'Legislação Penal e Proteção de Crianças', # penal, crime, crianca, adolescente, pena
    7: 'Radiodifusão Comunitária',                # radiodifusao comunitaria, municipio, associacao
    8: 'Requerimentos e Regimento Interno',       # regimento interno, senado, audiencia
    9: 'Créditos e Orçamento Federal',            # credito, republica federativa, abre, favor
}

pendentes = [c for c, nome in NOMES_CIDADAOS.items() if '(renomear)' in nome]
if pendentes:
    print(f'⚠️  Clusters ainda sem nome cidadão: {pendentes}')
else:
    print('Todos os clusters nomeados. ✓')

for c in range(len(NOMES_CIDADAOS)):
    print(f'  {c} → {NOMES_CIDADAOS[c]}')

Todos os clusters nomeados. ✓
  0 → Destaques e Requerimentos de Votação
  1 → Políticas Públicas e Programas Sociais
  2 → Licenças Parlamentares
  3 → Educação, Trânsito e Direitos Sociais
  4 → Sistema Único de Saúde (SUS)
  5 → Radiodifusão Sonora (Rádio FM/AM)
  6 → Legislação Penal e Proteção de Crianças
  7 → Radiodifusão Comunitária
  8 → Requerimentos e Regimento Interno
  9 → Créditos e Orçamento Federal


## 6. Gravar `tema_cluster` e `tema_cidadao` no Supabase

Reaproveita `upsert_proposicoes` (chave `id_externo, casa`): atualiza só as colunas
de tema, preservando ementa/keywords/data já gravadas na Sprint 1.

In [8]:
df['tema_cidadao'] = df['tema_cluster'].map(NOMES_CIDADAOS)

registros = [
    {
        'id_externo': int(r['id_externo']),
        'casa': r['casa'],
        'tema_cluster': int(r['tema_cluster']),
        'tema_cidadao': r['tema_cidadao'],
    }
    for _, r in df.iterrows()
]
print(f'Registros a atualizar: {len(registros)}')

total = upsert_proposicoes(registros)
print(f'Proposições atualizadas com tema: {total}')

Registros a atualizar: 22040


2026-06-23 22:21:32,434 [INFO] HTTP Request: POST https://woxluznsfqskopzzjqtf.supabase.co/rest/v1/proposicoes?on_conflict=id_externo%2Ccasa&columns=%22casa%22%2C%22id_externo%22%2C%22tema_cluster%22%2C%22tema_cidadao%22 "HTTP/2 200 OK"
2026-06-23 22:21:32,633 [INFO]   proposicoes: 500/22040
2026-06-23 22:21:34,482 [INFO] HTTP Request: POST https://woxluznsfqskopzzjqtf.supabase.co/rest/v1/proposicoes?on_conflict=id_externo%2Ccasa&columns=%22casa%22%2C%22id_externo%22%2C%22tema_cluster%22%2C%22tema_cidadao%22 "HTTP/2 200 OK"
2026-06-23 22:21:34,798 [INFO]   proposicoes: 1000/22040
2026-06-23 22:21:35,712 [INFO] HTTP Request: POST https://woxluznsfqskopzzjqtf.supabase.co/rest/v1/proposicoes?on_conflict=id_externo%2Ccasa&columns=%22casa%22%2C%22id_externo%22%2C%22tema_cluster%22%2C%22tema_cidadao%22 "HTTP/2 200 OK"
2026-06-23 22:21:35,721 [INFO]   proposicoes: 1500/22040
2026-06-23 22:21:36,704 [INFO] HTTP Request: POST https://woxluznsfqskopzzjqtf.supabase.co/rest/v1/proposicoes?on_confl

Proposições atualizadas com tema: 22040


## 7. Sanidade

In [9]:
verif = pd.DataFrame(buscar_todos('proposicoes', 'casa,tema_cluster,tema_cidadao'))

print('=== Distribuição por tema_cluster ===')
print(verif['tema_cluster'].value_counts(dropna=False).sort_index())

print('\n=== Proposições com tema_cidadao preenchido, por casa ===')
print(verif[verif['tema_cidadao'].notna()].groupby('casa').size())

print('\n=== Mapa cluster → nome cidadão ===')
print(
    verif[verif['tema_cidadao'].notna()]
    .groupby(['tema_cluster', 'tema_cidadao'])
    .size()
    .rename('proposicoes')
)

2026-06-23 22:23:21,610 [INFO] HTTP Request: GET https://woxluznsfqskopzzjqtf.supabase.co/rest/v1/proposicoes?select=casa%2Ctema_cluster%2Ctema_cidadao&offset=0&limit=1000 "HTTP/2 200 OK"
2026-06-23 22:23:21,984 [INFO] HTTP Request: GET https://woxluznsfqskopzzjqtf.supabase.co/rest/v1/proposicoes?select=casa%2Ctema_cluster%2Ctema_cidadao&offset=1000&limit=1000 "HTTP/2 200 OK"
2026-06-23 22:23:22,428 [INFO] HTTP Request: GET https://woxluznsfqskopzzjqtf.supabase.co/rest/v1/proposicoes?select=casa%2Ctema_cluster%2Ctema_cidadao&offset=2000&limit=1000 "HTTP/2 200 OK"
2026-06-23 22:23:23,236 [INFO] HTTP Request: GET https://woxluznsfqskopzzjqtf.supabase.co/rest/v1/proposicoes?select=casa%2Ctema_cluster%2Ctema_cidadao&offset=3000&limit=1000 "HTTP/2 200 OK"
2026-06-23 22:23:24,046 [INFO] HTTP Request: GET https://woxluznsfqskopzzjqtf.supabase.co/rest/v1/proposicoes?select=casa%2Ctema_cluster%2Ctema_cidadao&offset=4000&limit=1000 "HTTP/2 200 OK"
2026-06-23 22:23:24,573 [INFO] HTTP Request: GET

=== Distribuição por tema_cluster ===
tema_cluster
0.0      422
1.0    13685
2.0     1138
3.0     1732
4.0      381
5.0      916
6.0     1231
7.0      873
8.0     1263
9.0      399
NaN        1
Name: count, dtype: int64

=== Proposições com tema_cidadao preenchido, por casa ===
casa
camara    11697
senado    10343
dtype: int64

=== Mapa cluster → nome cidadão ===
tema_cluster  tema_cidadao                           
0.0           Destaques e Requerimentos de Votação         422
1.0           Políticas Públicas e Programas Sociais     13685
2.0           Licenças Parlamentares                      1138
3.0           Educação, Trânsito e Direitos Sociais       1732
4.0           Sistema Único de Saúde (SUS)                 381
5.0           Radiodifusão Sonora (Rádio FM/AM)            916
6.0           Legislação Penal e Proteção de Crianças     1231
7.0           Radiodifusão Comunitária                     873
8.0           Requerimentos e Regimento Interno           1263
9.0          